In [ ]:
import numpy as np

# -----------------------
# Order book
# -----------------------
bids = [
    (30, 30000),
    (29, 5000),
    (28, 12000),
    (27, 28000),
]

asks = [
    (28, 40000),
    (31, 20000),
    (32, 20000),
    (33, 30000),
]

RESALE_PRICE = 30

# -----------------------
# Clearing price
# -----------------------
def find_clearing_price(bids, asks):
    all_prices = sorted(set([p for p, _ in bids] + [p for p, _ in asks]))

    best_price = None
    best_volume = -1

    for p in all_prices:
        bid_vol = sum(v for price, v in bids if price >= p)
        ask_vol = sum(v for price, v in asks if price <= p)
        volume = min(bid_vol, ask_vol)

        if volume > best_volume or (volume == best_volume and (best_price is None or p > best_price)):
            best_volume = volume
            best_price = p

    return best_price, best_volume

# -----------------------
# Fill calculation (you are LAST in queue)
# -----------------------
def compute_fill(bids, asks, my_price, my_qty):
    bids_with_me = bids + [(my_price, my_qty)]
    clearing_price, total_volume = find_clearing_price(bids_with_me, asks)

    if my_price < clearing_price:
        return 0, clearing_price

    total_bid = sum(v for p, v in bids_with_me if p >= clearing_price)
    total_ask = sum(v for p, v in asks if p <= clearing_price)
    executed = min(total_bid, total_ask)

    # volume ahead of you
    ahead = 0
    for p, v in bids:
        if p > my_price and p >= clearing_price:
            ahead += v
        elif p == my_price and p >= clearing_price:
            ahead += v  # you're last at this level

    remaining = max(0, executed - ahead)
    my_fill = min(my_qty, remaining)

    return my_fill, clearing_price

# -----------------------
# Price grid (integers only)
# -----------------------
price_grid = sorted(set([p for p, _ in bids] + [p for p, _ in asks]))

# -----------------------
# Quantity grid (1000–20000 step 500 + ±1 refinement)
# -----------------------
base_qty_grid = np.arange(1000, 20001, 500)

qty_grid = set()
for q in base_qty_grid:
    qty_grid.add(q)
    if q - 1 > 0:
        qty_grid.add(q - 1)
    qty_grid.add(q + 1)

qty_grid = sorted(qty_grid)

# -----------------------
# Grid search + PnL
# -----------------------
results = []
best = None

for price in price_grid:
    for qty in qty_grid:
        fill, clearing_price = compute_fill(bids, asks, price, qty)

        cost = fill * clearing_price
        revenue = fill * RESALE_PRICE
        pnl = revenue - cost

        results.append({
            "price": price,
            "qty": qty,
            "fill": fill,
            "clearing_price": clearing_price,
            "cost": cost,
            "revenue": revenue,
            "pnl": pnl
        })

        if best is None or pnl > best["pnl"]:
            best = results[-1]

# -----------------------
# Output best result
# -----------------------
print("Best strategy:\n")
for k, v in best.items():
    print(f"{k}: {v}")

# -----------------------
# Top 10 strategies
# -----------------------
top = sorted(results, key=lambda x: x["pnl"], reverse=True)[:10]

print("\nTop 10 strategies:\n")
for r in top:
    print(r)

Best strategy:

price: 30
qty: 9999
fill: 9999
clearing_price: 29
cost: 289971
revenue: 299970
pnl: 9999

Top 10 strategies:

{'price': 30, 'qty': np.int64(9999), 'fill': np.int64(9999), 'clearing_price': 29, 'cost': np.int64(289971), 'revenue': np.int64(299970), 'pnl': np.int64(9999)}
{'price': 31, 'qty': np.int64(9999), 'fill': np.int64(9999), 'clearing_price': 29, 'cost': np.int64(289971), 'revenue': np.int64(299970), 'pnl': np.int64(9999)}
{'price': 32, 'qty': np.int64(9999), 'fill': np.int64(9999), 'clearing_price': 29, 'cost': np.int64(289971), 'revenue': np.int64(299970), 'pnl': np.int64(9999)}
{'price': 33, 'qty': np.int64(9999), 'fill': np.int64(9999), 'clearing_price': 29, 'cost': np.int64(289971), 'revenue': np.int64(299970), 'pnl': np.int64(9999)}
{'price': 29, 'qty': np.int64(4999), 'fill': np.int64(4999), 'clearing_price': 28, 'cost': np.int64(139972), 'revenue': np.int64(149970), 'pnl': np.int64(9998)}
{'price': 30, 'qty': np.int64(4999), 'fill': np.int64(4999), 'clearin

In [ ]:
# -----------------------
# Order book
# -----------------------
bids = [
    (20, 43000),
    (19, 17000),
    (18, 6000),
    (17, 5000),
    (16, 10000),
    (15, 5000),
    (14, 10000),
    (13, 7000),
]

asks = [
    (12, 20000),
    (13, 25000),
    (14, 35000),
    (15, 6000),
    (16, 5000),
    (17, 0),
    (18, 10000),
    (19, 12000),
]

RESALE_PRICE = 20
FEE_PER_UNIT = 0.05

# -----------------------
# Clearing price
# -----------------------
def find_clearing_price(bids, asks):
    all_prices = sorted(set([p for p, _ in bids] + [p for p, _ in asks]))

    best_price = None
    best_volume = -1

    for p in all_prices:
        bid_vol = sum(v for price, v in bids if price >= p)
        ask_vol = sum(v for price, v in asks if price <= p)
        volume = min(bid_vol, ask_vol)

        if volume > best_volume or (volume == best_volume and (best_price is None or p > best_price)):
            best_volume = volume
            best_price = p

    return best_price, best_volume

# -----------------------
# Fill calculation
# -----------------------
def compute_fill(bids, asks, my_price, my_qty):
    bids_with_me = bids + [(my_price, my_qty)]
    clearing_price, total_volume = find_clearing_price(bids_with_me, asks)

    if my_price < clearing_price:
        return 0, clearing_price

    total_bid = sum(v for p, v in bids_with_me if p >= clearing_price)
    total_ask = sum(v for p, v in asks if p <= clearing_price)
    executed = min(total_bid, total_ask)

    # queue ahead of you
    ahead = 0
    for p, v in bids:
        if p > my_price and p >= clearing_price:
            ahead += v
        elif p == my_price and p >= clearing_price:
            ahead += v

    remaining = max(0, executed - ahead)
    my_fill = min(my_qty, remaining)

    return my_fill, clearing_price

# -----------------------
# Exhaustive search
# -----------------------
price_grid = sorted(set([p for p, _ in bids] + [p for p, _ in asks]))
qty_grid = range(0, 25001)  # FULL search

best = None

for price in price_grid:
    for qty in qty_grid:
        fill, clearing_price = compute_fill(bids, asks, price, qty)

        cost = fill * clearing_price
        revenue = fill * RESALE_PRICE
        fees = fill * (2 * FEE_PER_UNIT)
        pnl = revenue - cost - fees

        if best is None or pnl > best["pnl"]:
            best = {
                "price": price,
                "qty": qty,
                "fill": fill,
                "clearing_price": clearing_price,
                "cost": cost,
                "revenue": revenue,
                "fees": fees,
                "pnl": pnl
            }

# -----------------------
# Output
# -----------------------
print("Best strategy:\n")
for k, v in best.items():
    print(f"{k}: {v}")

Best strategy:

price: 17
qty: 19999
fill: 19999
clearing_price: 16
cost: 319984
revenue: 399980
fees: 1999.9
pnl: 77996.1


In [ ]:
import math

BOOKS = {
    "DRYLAND_FLAX": {
        "bids": [(30, 30000), (29, 5000), (28, 12000), (27, 28000)],
        "asks": [(28, 40000), (31, 20000), (32, 20000), (33, 30000)],
        "resale_price": 30,
        "fee_per_side": 0.00,
    },
    "EMBER_MUSHROOM": {
        "bids": [(20, 43000), (19, 17000), (18, 6000), (17, 5000),
                 (16, 10000), (15, 5000), (14, 10000), (13, 7000)],
        "asks": [(12, 20000), (13, 25000), (14, 35000), (15, 6000),
                 (16, 5000), (17, 0), (18, 10000), (19, 12000)],
        "resale_price": 20,
        "fee_per_side": 0.05,
    }
}

def exhaustive_best_bid(bids, asks, resale_price, fee_per_side=0.0):
    """
    Exhaustive search over all relevant integer bid prices and quantities.
    Assumes:
      - you submit one BID order
      - you are last in queue at your price
      - auction chooses clearing price that maximizes traded volume,
        then breaks ties by choosing the HIGHER price
    """

    # No reason to search above the highest relevant book/resale price.
    max_price = max(
        max(p for p, _ in bids),
        max(p for p, _ in asks),
        int(math.ceil(resale_price))
    )

    # No reason to search quantities above total ask depth:
    # you can never get filled on more than that.
    max_qty = sum(v for _, v in asks)

    qtys = np.arange(max_qty + 1, dtype=np.int64)

    best_pnl = -float("inf")
    best_rows = []

    for my_price in range(max_price + 1):
        # Candidate clearing prices are book price levels + your own price.
        clearing_prices = sorted(set(
            [p for p, _ in bids] + [p for p, _ in asks] + [my_price]
        ))

        # Precompute base cumulative volumes and queue-ahead values at each clearing price.
        base_bid = {}
        base_ask = {}
        ahead = {}

        for cp in clearing_prices:
            base_bid[cp] = sum(v for p, v in bids if p >= cp)
            base_ask[cp] = sum(v for p, v in asks if p <= cp)

            # You are last at your own price level.
            ahead[cp] = (
                sum(v for p, v in bids if p > my_price and p >= cp)
                + sum(v for p, v in bids if p == my_price and p >= cp)
            )

        # For each quantity, determine the auction clearing price.
        # We use >= so ties go to the HIGHER clearing price.
        best_cp = np.full(max_qty + 1, clearing_prices[0], dtype=np.int64)
        best_vol = np.full(max_qty + 1, -1, dtype=np.int64)

        for cp in clearing_prices:  # ascending + >= => higher price wins ties
            if my_price >= cp:
                bid_curve = base_bid[cp] + qtys
            else:
                bid_curve = np.full(max_qty + 1, base_bid[cp], dtype=np.int64)

            vol_curve = np.minimum(bid_curve, base_ask[cp])

            mask = vol_curve >= best_vol
            best_vol[mask] = vol_curve[mask]
            best_cp[mask] = cp

        # Compute your fill and pnl for each quantity under the chosen clearing price.
        fill = np.zeros(max_qty + 1, dtype=np.int64)
        pnl = np.zeros(max_qty + 1, dtype=np.float64)

        for cp in np.unique(best_cp):
            idx = np.where(best_cp == cp)[0]

            if my_price < cp:
                continue  # your bid does not participate

            executed = np.minimum(base_bid[cp] + idx, base_ask[cp])
            my_fill = np.minimum(idx, np.maximum(0, executed - ahead[cp]))

            fill[idx] = my_fill
            pnl[idx] = my_fill * (resale_price - cp - 2 * fee_per_side)

        local_best = pnl.max()

        if local_best > best_pnl + 1e-12:
            best_pnl = local_best
            best_rows = []

        if abs(local_best - best_pnl) <= 1e-12:
            best_qs = np.where(np.abs(pnl - best_pnl) <= 1e-12)[0]
            for q in best_qs:
                best_rows.append({
                    "price": int(my_price),
                    "qty": int(q),
                    "fill": int(fill[q]),
                    "clearing_price": int(best_cp[q]),
                    "pnl": float(pnl[q]),
                })

    # Sort tied optima for readability
    best_rows = sorted(best_rows, key=lambda r: (r["price"], r["qty"]))

    # Practical recommendation:
    # among tied optima, prefer the highest price that is <= resale price
    # (this gives 30 for Dryland and 20 for Ember here).
    candidates = [r for r in best_rows if r["price"] <= resale_price]
    if candidates:
        recommended = max(candidates, key=lambda r: (r["price"], -r["qty"]))
    else:
        recommended = max(best_rows, key=lambda r: (r["price"], -r["qty"]))

    return {
        "best_pnl": best_pnl,
        "recommended": recommended,
        "all_tied_optima": best_rows,
    }

# Run both books
for name, book in BOOKS.items():
    out = exhaustive_best_bid(
        bids=book["bids"],
        asks=book["asks"],
        resale_price=book["resale_price"],
        fee_per_side=book["fee_per_side"],
    )

    print(f"\n{'='*70}")
    print(name)
    print(f"{'='*70}")
    print("Recommended submission:")
    print(out["recommended"])
    print(f"\nBest PnL: {out['best_pnl']:.2f}")
    print("\nAll tied optimal orders:")
    for row in out["all_tied_optima"]:
        print(row)


DRYLAND_FLAX
Recommended submission:
{'price': 30, 'qty': 9999, 'fill': 9999, 'clearing_price': 29, 'pnl': 9999.0}

Best PnL: 9999.00

All tied optimal orders:
{'price': 30, 'qty': 9999, 'fill': 9999, 'clearing_price': 29, 'pnl': 9999.0}
{'price': 31, 'qty': 9999, 'fill': 9999, 'clearing_price': 29, 'pnl': 9999.0}
{'price': 32, 'qty': 9999, 'fill': 9999, 'clearing_price': 29, 'pnl': 9999.0}
{'price': 33, 'qty': 9999, 'fill': 9999, 'clearing_price': 29, 'pnl': 9999.0}

EMBER_MUSHROOM
Recommended submission:
{'price': 20, 'qty': 19999, 'fill': 19999, 'clearing_price': 16, 'pnl': 77996.09999999999}

Best PnL: 77996.10

All tied optimal orders:
{'price': 17, 'qty': 19999, 'fill': 19999, 'clearing_price': 16, 'pnl': 77996.09999999999}
{'price': 18, 'qty': 19999, 'fill': 19999, 'clearing_price': 16, 'pnl': 77996.09999999999}
{'price': 19, 'qty': 19999, 'fill': 19999, 'clearing_price': 16, 'pnl': 77996.09999999999}
{'price': 20, 'qty': 19999, 'fill': 19999, 'clearing_price': 16, 'pnl': 77996